# Dataset selection: picking a real, illuminated LROC WAC EDR to test with

Queries the real LROC catalog (via the PDS Geosciences Node ODE REST API) for a multi-orbit
window with favorable illumination geometry and good WAC data availability, and picks one real
EDR product to use as this demo's test image -- see `../docs/plan.md`/`trntest.dataset` for the
full selection approach (SPICE-derived orbit/illumination geometry, throttling, per-orbit
coverage).

This notebook's only job is selection: it writes the full candidate table to a small, checked-in
CSV (`dataset_manifest.csv`, alongside this notebook) and stops -- no DEM/ortho fetch, no
`sat_sim` render. `../notebooks/image_generation.ipynb` reads that checked-in file and does the
rest, so the two notebooks never need to run in the same session: rerun this one and commit an
updated `dataset_manifest.csv` to change which real image the demo renders.

In [1]:
import trntest

session = trntest.Session()

## Catalog-driven selection

`session.select_dataset()` returns a throttled, illumination-filtered list of real EDR
candidates (one row per image) for the chosen search window, including everything needed to
relocate and pose each one later: `edr_volume`/`edr_subdir`/`edr_doy`/`edr_product`,
`cdr_volume`/`cdr_product`, and `start_frame`.

In [2]:
images = session.select_dataset(max_search_days=7)
images

select_dataset: 81 images across 12 orbits (2019-11-01 01:13:09.847000+00:00 to 2019-11-02 00:40:33.176000+00:00); per-orbit counts: {46625: 7, 46626: 7, 46627: 7, 46628: 7, 46629: 7, 46630: 7, 46631: 7, 46632: 7, 46633: 7, 46634: 7, 46635: 7, 46636: 7}


,product_id,edr_volume,edr_subdir,edr_doy,edr_product,cdr_volume,cdr_subdir,cdr_doy,cdr_product,orbit_number,start_time,stop_time,start_frame,center_frame_index,n_frames_for_square_crop,sun_elevation_deg,incidence_angle_deg,center_lat_deg,center_lon_deg
0,M1327210646CE,LROLRC_0041B,ESM4,2019305,M1327210646CE,LROLRC_1041B,ESM4,2019305,M1327210646CC,46625,2019-11-01 01:22:59.051000+00:00,2019-11-01 01:29:01.864000+00:00,93.5,129.0,71,41.844544,48.16,38.5020,169.5177
1,M1327211014CE,LROLRC_0041B,ESM4,2019305,M1327211014CE,LROLRC_1041B,ESM4,2019305,M1327210646CC,46625,2019-11-01 01:29:06.171000+00:00,2019-11-01 01:34:22.421000+00:00,75.0,110.0,70,28.650836,61.35,55.4504,171.8937
2,M1327211334CE,LROLRC_0041B,ESM4,2019305,M1327211334CE,LROLRC_1041B,ESM4,2019305,M1327210646CC,46625,2019-11-01 01:34:26.729000+00:00,2019-11-01 01:39:25.323000+00:00,69.5,105.0,71,15.823285,74.18,70.7112,177.0545
3,M1327215525CE,LROLRC_0041B,ESM4,2019305,M1327215525CE,LROLRC_1041B,ESM4,2019305,M1327215170CC,46625,2019-11-01 02:44:17.189000+00:00,2019-11-01 02:52:24.126000+00:00,259.5,294.0,69,19.388089,70.61,-67.5433,156.7774
4,M1327216016CE,LROLRC_0041B,ESM4,2019305,M1327216016CE,LROLRC_1041B,ESM4,2019305,M1327215525CC,46625,2019-11-01 02:52:28.428000+00:00,2019-11-01 02:57:51.396000+00:00,126.5,159.0,65,36.649340,53.35,-46.4285,161.8946
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,M1327288708CE,LROLRC_0041B,ESM4,2019305,M1327288708CE,LROLRC_1041B,ESM4,2019305,M1327288003CC,46636,2019-11-01 23:04:00.714000+00:00,2019-11-01 23:10:23.683000+00:00,94.0,129.0,70,15.866035,74.14,70.9007,165.3437
77,M1327292971CE,LROLRC_0041B,ESM4,2019306,M1327292971CE,LROLRC_1041B,ESM4,2019306,M1327292540CC,46636,2019-11-02 00:15:03.961000+00:00,2019-11-02 00:22:22.711000+00:00,227.0,260.0,66,20.268884,69.73,-66.7155,145.0769
78,M1327293414CE,LROLRC_0041B,ESM4,2019306,M1327293414CE,LROLRC_1041B,ESM4,2019306,M1327292971CC,46636,2019-11-02 00:22:27.016000+00:00,2019-11-02 00:27:43.454000+00:00,132.0,166.0,68,36.628572,53.37,-46.9781,149.8148
79,M1327293735CE,LROLRC_0041B,ESM4,2019306,M1327293735CE,LROLRC_1041B,ESM4,2019306,M1327292971CC,46636,2019-11-02 00:27:47.758000+00:00,2019-11-02 00:36:07.133000+00:00,199.0,235.0,72,51.669499,38.33,-25.9000,151.8709


## Selected image + checked-in manifest

`image_generation.ipynb` only ever renders the first row of the manifest
(`generate_dataset(images, limit=1)`), so that's the "selected" image below. Writing the *full*
candidate table (not just this one row) to `dataset_manifest.csv` also documents what the rest
of this search window looked like.

In [3]:
selected = images.iloc[0]
print(f"Selected EDR product: {selected['edr_product']}")
print(
    f"  edr_volume={selected['edr_volume']} edr_subdir={selected['edr_subdir']} "
    f"edr_doy={selected['edr_doy']} start_frame={selected['start_frame']}"
)

manifest_path = "dataset_manifest.csv"
trntest.write_manifest(images, manifest_path)
print(f"Wrote {len(images)} candidate row(s) to {manifest_path}")

Selected EDR product: M1327210646CE
  edr_volume=LROLRC_0041B edr_subdir=ESM4 edr_doy=2019305 start_frame=93.5
Wrote 81 candidate row(s) to dataset_manifest.csv


## Dataset folder (cheap convenience/validation step)

`trntest.TrnTestDataSet.create()` sets up (or reuses) the self-contained `trn_dataset` folder
`image_generation.ipynb` actually populates and reads from -- see `../docs/dataset-plan.md` for
the full design. This call is idempotent and never touches already-generated product files, so
it's safe to run here purely to confirm the full candidate table round-trips into a valid dataset
folder; `image_generation.ipynb`'s own call is the authoritative one (it has no runtime
dependency on this notebook having run first, same as the manifest CSV above).

In [4]:
trntest.TrnTestDataSet.create(session.config.output_dir / "trn_dataset", images, session.config)
print(f"Dataset folder ready at {session.config.output_dir / 'trn_dataset'}")

Dataset folder ready at /workspace/output/trn_dataset
